# Part 1 — Synthetic GANs from Scratch

This notebook reproduces the tutorial sine-wave GAN and extends it with a harder **2D spiral** dataset.
It also compares a **baseline GAN** against an **improved GAN** with deeper layers and LeakyReLU activations.

## Notebook goals
1. Reproduce the sine-wave GAN from the tutorial.
2. Build a second synthetic dataset using a 2D spiral.
3. Modify the architecture and compare baseline vs improved GAN.
4. Save clean, report-ready plots and CSV summaries.

## Colab notes
- Upload this notebook to Google Colab.
- Run all cells in order.
- All outputs are saved into:
  - `outputs/part1/csv`
  - `outputs/part1/plots`

In [ ]:
# ============================================================
# CELL 1 — IMPORTS, PATHS, REPRODUCIBILITY, AND VISUAL STYLE
# ============================================================
"""
Import all required libraries, create output folders, and define a
consistent plotting style for the whole notebook.
"""

from pathlib import Path
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

def set_seed(seed: int = 42) -> None:
    """
    Set random seeds for Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# -----------------------------
# Device
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -----------------------------
# Output folders
# -----------------------------
BASE_OUTPUT_DIR = Path("outputs") / "part1"
CSV_DIR = BASE_OUTPUT_DIR / "csv"
PLOTS_DIR = BASE_OUTPUT_DIR / "plots"

for folder in [BASE_OUTPUT_DIR, CSV_DIR, PLOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"CSV outputs will be saved to: {CSV_DIR.resolve()}")
print(f"Plot outputs will be saved to: {PLOTS_DIR.resolve()}")

# -----------------------------
# Plot style (avoid blue)
# -----------------------------
COLOR_REAL = "#7A1F1F"        # deep red
COLOR_FAKE = "#C97B63"        # warm clay
COLOR_ALT = "#6C8A3B"         # olive green
COLOR_GRID = "#D9D2C3"        # soft sand
COLOR_TEXT = "#2F2A24"        # charcoal

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.facecolor"] = "#FAF7F2"
plt.rcParams["figure.facecolor"] = "#FFFFFF"
plt.rcParams["axes.edgecolor"] = "#B9AD99"
plt.rcParams["grid.color"] = COLOR_GRID
plt.rcParams["axes.labelcolor"] = COLOR_TEXT
plt.rcParams["xtick.color"] = COLOR_TEXT
plt.rcParams["ytick.color"] = COLOR_TEXT
plt.rcParams["text.color"] = COLOR_TEXT
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.grid"] = True

def save_plot(filename: str) -> None:
    """
    Save the current matplotlib figure into the part1 plot directory.
    """
    filepath = PLOTS_DIR / filename
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    print(f"Saved plot: {filepath}")

In [ ]:
# ============================================================
# CELL 2 — HELPER FUNCTIONS FOR DATA GENERATION
# ============================================================
"""
Create the synthetic datasets used in this notebook.

The sine-wave data reproduces the tutorial example.
The spiral data acts as a harder nonlinear 2D target distribution.
"""

def create_sine_wave_data(n_samples: int = 1024) -> np.ndarray:
    """
    Create the tutorial-style 2D sine-wave dataset.

    x1 is sampled uniformly from [0, 2pi].
    x2 is sin(x1).
    """
    x1 = 2.0 * math.pi * np.random.rand(n_samples)
    x2 = np.sin(x1)
    data = np.column_stack([x1, x2]).astype(np.float32)
    return data

def create_spiral_data(n_samples: int = 1024, noise_std: float = 0.08) -> np.ndarray:
    """
    Create a noisy 2D spiral dataset.

    The spiral is more challenging than the sine-wave because the data
    wraps around the plane and requires the GAN to capture a curved,
    non-monotonic structure.
    """
    t = np.linspace(0.5, 4.5 * math.pi, n_samples)
    r = np.linspace(0.1, 1.0, n_samples)

    x = r * np.cos(t)
    y = r * np.sin(t)

    x += np.random.normal(0, noise_std, size=n_samples)
    y += np.random.normal(0, noise_std, size=n_samples)

    data = np.column_stack([x, y]).astype(np.float32)
    return data

def make_loader(data: np.ndarray, batch_size: int = 32) -> DataLoader:
    """
    Convert a NumPy array into a shuffled PyTorch DataLoader.
    """
    tensor_x = torch.tensor(data, dtype=torch.float32)
    tensor_y = torch.zeros(len(data), dtype=torch.float32)
    dataset = TensorDataset(tensor_x, tensor_y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

def summarise_dataset(data: np.ndarray, dataset_name: str) -> pd.DataFrame:
    """
    Create a summary table for a 2D dataset.
    """
    df = pd.DataFrame(data, columns=["x1", "x2"])
    summary = df.describe().T.reset_index().rename(columns={"index": "feature"})
    summary.insert(0, "dataset_name", dataset_name)
    return summary

In [ ]:
# ============================================================
# CELL 3 — CREATE AND INSPECT THE SYNTHETIC DATASETS
# ============================================================
"""
Generate both synthetic datasets and save their summary tables.
"""

TRAIN_DATA_LENGTH = 1024
BATCH_SIZE = 32

sine_data = create_sine_wave_data(n_samples=TRAIN_DATA_LENGTH)
spiral_data = create_spiral_data(n_samples=TRAIN_DATA_LENGTH, noise_std=0.08)

sine_loader = make_loader(sine_data, batch_size=BATCH_SIZE)
spiral_loader = make_loader(spiral_data, batch_size=BATCH_SIZE)

sine_summary = summarise_dataset(sine_data, "sine_wave")
spiral_summary = summarise_dataset(spiral_data, "spiral")

dataset_summary = pd.concat([sine_summary, spiral_summary], ignore_index=True)
dataset_summary.to_csv(CSV_DIR / "01_dataset_summary_statistics.csv", index=False)

print("Dataset summary saved.")
display(dataset_summary.head(10))

In [ ]:
# ============================================================
# CELL 4 — PLOT THE REAL SYNTHETIC DATA
# ============================================================
"""
Visualise the real sine-wave and spiral data before GAN training.
"""

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(sine_data[:, 0], sine_data[:, 1], s=14, alpha=0.75, color=COLOR_REAL)
axes[0].set_title("Real Sine-Wave Data")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

axes[1].scatter(spiral_data[:, 0], spiral_data[:, 1], s=14, alpha=0.75, color=COLOR_ALT)
axes[1].set_title("Real Spiral Data")
axes[1].set_xlabel("x1")
axes[1].set_ylabel("x2")

save_plot("02_real_synthetic_datasets.png")
plt.show()

In [ ]:
# ============================================================
# CELL 5 — GAN MODEL DEFINITIONS
# ============================================================
"""
Define flexible generator and discriminator classes.

Two model variants will be used:
1. Baseline GAN: close to the tutorial style.
2. Improved GAN: deeper and uses LeakyReLU.
"""

class Generator(nn.Module):
    """
    Flexible MLP generator for 2D synthetic data.
    """
    def __init__(self, latent_dim: int, hidden_dims: list[int], activation: str = "relu"):
        super().__init__()

        activation_layer = nn.ReLU if activation.lower() == "relu" else nn.LeakyReLU
        layers = []
        in_dim = latent_dim

        for hidden_dim in hidden_dims:
            if activation.lower() == "relu":
                layers.extend([
                    nn.Linear(in_dim, hidden_dim),
                    activation_layer(),
                ])
            else:
                layers.extend([
                    nn.Linear(in_dim, hidden_dim),
                    activation_layer(0.2),
                ])
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, 2))
        self.model = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.model(z)


class Discriminator(nn.Module):
    """
    Flexible MLP discriminator for 2D synthetic data.
    """
    def __init__(
        self,
        input_dim: int = 2,
        hidden_dims: list[int] = [256, 128, 64],
        activation: str = "relu",
        dropout: float = 0.3
    ):
        super().__init__()

        activation_layer = nn.ReLU if activation.lower() == "relu" else nn.LeakyReLU
        layers = []
        in_dim = input_dim

        for hidden_dim in hidden_dims:
            if activation.lower() == "relu":
                layers.extend([
                    nn.Linear(in_dim, hidden_dim),
                    activation_layer(),
                    nn.Dropout(dropout),
                ])
            else:
                layers.extend([
                    nn.Linear(in_dim, hidden_dim),
                    activation_layer(0.2),
                    nn.Dropout(dropout),
                ])
            in_dim = hidden_dim

        layers.extend([
            nn.Linear(in_dim, 1),
            nn.Sigmoid(),
        ])

        self.model = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

In [ ]:
# ============================================================
# CELL 6 — MODEL CONFIGURATION FACTORIES
# ============================================================
"""
Create baseline and improved GAN configurations in a clean way.
"""

LATENT_DIM = 2
LEARNING_RATE = 0.001
NUM_EPOCHS = 2000

def build_baseline_gan(latent_dim: int = LATENT_DIM):
    """
    Baseline GAN configuration based on the tutorial.
    """
    generator = Generator(
        latent_dim=latent_dim,
        hidden_dims=[16, 32],
        activation="relu",
    ).to(DEVICE)

    discriminator = Discriminator(
        input_dim=2,
        hidden_dims=[256, 128, 64],
        activation="relu",
        dropout=0.3,
    ).to(DEVICE)

    return generator, discriminator

def build_improved_gan(latent_dim: int = LATENT_DIM):
    """
    Improved GAN with a deeper generator and LeakyReLU activations.

    This architecture is designed to provide a smoother learning signal
    and more expressive transformations for the harder spiral structure.
    """
    generator = Generator(
        latent_dim=latent_dim,
        hidden_dims=[32, 64, 64],
        activation="leakyrelu",
    ).to(DEVICE)

    discriminator = Discriminator(
        input_dim=2,
        hidden_dims=[256, 128, 64],
        activation="leakyrelu",
        dropout=0.2,
    ).to(DEVICE)

    return generator, discriminator

In [ ]:
# ============================================================
# CELL 7 — GAN TRAINING AND SAMPLING UTILITIES
# ============================================================
"""
Implement the GAN training loop and helper functions for sampling
and basic quantitative comparison.
"""

def sample_generator(generator: nn.Module, n_samples: int, latent_dim: int) -> np.ndarray:
    """
    Generate synthetic samples from the trained generator.
    """
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, latent_dim, device=DEVICE)
        generated = generator(z).cpu().numpy()
    return generated

def calculate_distribution_summary_distance(real_data: np.ndarray, fake_data: np.ndarray) -> dict:
    """
    Calculate simple distribution-matching statistics for 2D synthetic data.

    This is not a formal GAN metric like FID, but it provides a useful
    low-dimensional quantitative comparison for Part 1.
    """
    real_mean = real_data.mean(axis=0)
    fake_mean = fake_data.mean(axis=0)

    real_std = real_data.std(axis=0)
    fake_std = fake_data.std(axis=0)

    mean_l2_distance = float(np.linalg.norm(real_mean - fake_mean))
    std_l2_distance = float(np.linalg.norm(real_std - fake_std))

    return {
        "real_mean_x1": float(real_mean[0]),
        "real_mean_x2": float(real_mean[1]),
        "fake_mean_x1": float(fake_mean[0]),
        "fake_mean_x2": float(fake_mean[1]),
        "real_std_x1": float(real_std[0]),
        "real_std_x2": float(real_std[1]),
        "fake_std_x1": float(fake_std[0]),
        "fake_std_x2": float(fake_std[1]),
        "mean_l2_distance": mean_l2_distance,
        "std_l2_distance": std_l2_distance,
    }

def train_gan(
    data_loader: DataLoader,
    build_model_fn,
    model_name: str,
    dataset_name: str,
    latent_dim: int = LATENT_DIM,
    num_epochs: int = NUM_EPOCHS,
    learning_rate: float = LEARNING_RATE,
    print_every: int = 200,
):
    """
    Train a GAN using the standard alternating optimization scheme.

    The discriminator is trained to distinguish real and generated samples.
    The generator is trained to fool the discriminator into predicting real.
    """
    set_seed(SEED)

    generator, discriminator = build_model_fn()

    loss_function = nn.BCELoss()
    optimizer_discriminator = optim.Adam(discriminator.parameters(), lr=learning_rate)
    optimizer_generator = optim.Adam(generator.parameters(), lr=learning_rate)

    history = []
    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0
        batches = 0

        for real_samples, _ in data_loader:
            real_samples = real_samples.to(DEVICE)
            batch_size = real_samples.size(0)

            # -----------------------------
            # Train discriminator
            # -----------------------------
            real_labels = torch.ones((batch_size, 1), device=DEVICE)
            fake_labels = torch.zeros((batch_size, 1), device=DEVICE)

            latent_space_samples = torch.randn((batch_size, latent_dim), device=DEVICE)
            generated_samples = generator(latent_space_samples)

            all_samples = torch.cat((real_samples, generated_samples), dim=0)
            all_labels = torch.cat((real_labels, fake_labels), dim=0)

            discriminator.zero_grad()
            output_discriminator = discriminator(all_samples)
            loss_discriminator = loss_function(output_discriminator, all_labels)
            loss_discriminator.backward()
            optimizer_discriminator.step()

            # -----------------------------
            # Train generator
            # -----------------------------
            latent_space_samples = torch.randn((batch_size, latent_dim), device=DEVICE)

            generator.zero_grad()
            generated_samples = generator(latent_space_samples)
            output_discriminator_generated = discriminator(generated_samples)

            loss_generator = loss_function(output_discriminator_generated, real_labels)
            loss_generator.backward()
            optimizer_generator.step()

            epoch_d_loss += loss_discriminator.item()
            epoch_g_loss += loss_generator.item()
            batches += 1

        avg_d_loss = epoch_d_loss / batches
        avg_g_loss = epoch_g_loss / batches

        history.append({
            "dataset_name": dataset_name,
            "model_name": model_name,
            "epoch": epoch,
            "loss_discriminator": avg_d_loss,
            "loss_generator": avg_g_loss,
        })

        if epoch == 1 or epoch % print_every == 0 or epoch == num_epochs:
            print(
                f"[{dataset_name} | {model_name}] "
                f"Epoch {epoch:4d}/{num_epochs} | "
                f"Loss D: {avg_d_loss:.4f} | Loss G: {avg_g_loss:.4f}"
            )

    elapsed_seconds = time.time() - start_time
    history_df = pd.DataFrame(history)
    fake_samples = sample_generator(generator, n_samples=1000, latent_dim=latent_dim)

    metrics = calculate_distribution_summary_distance(
        real_data=next(iter([data_loader.dataset.tensors[0].numpy()])) ,
        fake_data=fake_samples
    )
    metrics["dataset_name"] = dataset_name
    metrics["model_name"] = model_name
    metrics["training_time_seconds"] = elapsed_seconds

    metrics_df = pd.DataFrame([metrics])

    return generator, discriminator, history_df, fake_samples, metrics_df

In [ ]:
# ============================================================
# CELL 8 — TRAIN BOTH GAN VARIANTS ON THE SINE-WAVE DATA
# ============================================================
"""
Train the baseline and improved GANs on the sine-wave dataset.
"""

sine_baseline_generator, sine_baseline_discriminator, sine_baseline_history, sine_baseline_fake, sine_baseline_metrics = train_gan(
    data_loader=sine_loader,
    build_model_fn=build_baseline_gan,
    model_name="baseline_gan",
    dataset_name="sine_wave",
    latent_dim=LATENT_DIM,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    print_every=200,
)

sine_improved_generator, sine_improved_discriminator, sine_improved_history, sine_improved_fake, sine_improved_metrics = train_gan(
    data_loader=sine_loader,
    build_model_fn=build_improved_gan,
    model_name="improved_gan",
    dataset_name="sine_wave",
    latent_dim=LATENT_DIM,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    print_every=200,
)

In [ ]:
# ============================================================
# CELL 9 — TRAIN BOTH GAN VARIANTS ON THE SPIRAL DATA
# ============================================================
"""
Train the baseline and improved GANs on the spiral dataset.
"""

spiral_baseline_generator, spiral_baseline_discriminator, spiral_baseline_history, spiral_baseline_fake, spiral_baseline_metrics = train_gan(
    data_loader=spiral_loader,
    build_model_fn=build_baseline_gan,
    model_name="baseline_gan",
    dataset_name="spiral",
    latent_dim=LATENT_DIM,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    print_every=200,
)

spiral_improved_generator, spiral_improved_discriminator, spiral_improved_history, spiral_improved_fake, spiral_improved_metrics = train_gan(
    data_loader=spiral_loader,
    build_model_fn=build_improved_gan,
    model_name="improved_gan",
    dataset_name="spiral",
    latent_dim=LATENT_DIM,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    print_every=200,
)

In [ ]:
# ============================================================
# CELL 10 — SAVE TRAINING HISTORIES AND SUMMARY METRICS
# ============================================================
"""
Combine and save all history and quantitative comparison tables.
"""

all_histories = pd.concat([
    sine_baseline_history,
    sine_improved_history,
    spiral_baseline_history,
    spiral_improved_history,
], ignore_index=True)

all_metrics = pd.concat([
    sine_baseline_metrics,
    sine_improved_metrics,
    spiral_baseline_metrics,
    spiral_improved_metrics,
], ignore_index=True)

all_histories.to_csv(CSV_DIR / "02_training_loss_history.csv", index=False)
all_metrics.to_csv(CSV_DIR / "03_quantitative_comparison_metrics.csv", index=False)

print("Saved training history and comparison metrics.")
display(all_metrics)

In [ ]:
# ============================================================
# CELL 11 — PLOT TRAINING LOSS CURVES
# ============================================================
"""
Plot discriminator and generator losses for each dataset-model pair.
"""

def plot_loss_curves(history_df: pd.DataFrame, dataset_name: str) -> None:
    """
    Plot discriminator and generator losses for one dataset.
    """
    subset = history_df[history_df["dataset_name"] == dataset_name].copy()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for model_name, color in [("baseline_gan", COLOR_REAL), ("improved_gan", COLOR_ALT)]:
        model_subset = subset[subset["model_name"] == model_name]

        axes[0].plot(
            model_subset["epoch"],
            model_subset["loss_discriminator"],
            label=model_name,
            linewidth=2,
            color=color,
        )
        axes[1].plot(
            model_subset["epoch"],
            model_subset["loss_generator"],
            label=model_name,
            linewidth=2,
            color=color,
        )

    axes[0].set_title(f"Discriminator Loss — {dataset_name.replace('_', ' ').title()}")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].set_title(f"Generator Loss — {dataset_name.replace('_', ' ').title()}")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    save_plot(f"03_{dataset_name}_loss_curves.png")
    plt.show()

plot_loss_curves(all_histories, "sine_wave")
plot_loss_curves(all_histories, "spiral")

In [ ]:
# ============================================================
# CELL 12 — VISUAL COMPARISON: REAL VS GENERATED SAMPLES
# ============================================================
"""
Plot real data against generated data for both baseline and improved GANs.
"""

def plot_real_vs_fake(real_data: np.ndarray, fake_data: np.ndarray, title: str, filename: str) -> None:
    """
    Plot real and generated samples side by side for visual comparison.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(real_data[:, 0], real_data[:, 1], s=12, alpha=0.75, color=COLOR_REAL)
    axes[0].set_title(f"Real Samples — {title}")
    axes[0].set_xlabel("x1")
    axes[0].set_ylabel("x2")

    axes[1].scatter(fake_data[:, 0], fake_data[:, 1], s=12, alpha=0.75, color=COLOR_FAKE)
    axes[1].set_title(f"Generated Samples — {title}")
    axes[1].set_xlabel("x1")
    axes[1].set_ylabel("x2")

    save_plot(filename)
    plt.show()

plot_real_vs_fake(
    real_data=sine_data,
    fake_data=sine_baseline_fake,
    title="Sine-Wave (Baseline GAN)",
    filename="04_sine_wave_real_vs_generated_baseline.png",
)

plot_real_vs_fake(
    real_data=sine_data,
    fake_data=sine_improved_fake,
    title="Sine-Wave (Improved GAN)",
    filename="05_sine_wave_real_vs_generated_improved.png",
)

plot_real_vs_fake(
    real_data=spiral_data,
    fake_data=spiral_baseline_fake,
    title="Spiral (Baseline GAN)",
    filename="06_spiral_real_vs_generated_baseline.png",
)

plot_real_vs_fake(
    real_data=spiral_data,
    fake_data=spiral_improved_fake,
    title="Spiral (Improved GAN)",
    filename="07_spiral_real_vs_generated_improved.png",
)

In [ ]:
# ============================================================
# CELL 13 — OVERLAY COMPARISON PLOTS
# ============================================================
"""
Overlay real and generated samples in the same plot to make
distribution matching easier to judge visually.
"""

def plot_overlay(real_data: np.ndarray, fake_data: np.ndarray, title: str, filename: str) -> None:
    """
    Overlay real and generated samples in a single scatter plot.
    """
    plt.figure(figsize=(7, 6))
    plt.scatter(real_data[:, 0], real_data[:, 1], s=14, alpha=0.55, color=COLOR_REAL, label="Real")
    plt.scatter(fake_data[:, 0], fake_data[:, 1], s=14, alpha=0.55, color=COLOR_FAKE, label="Generated")
    plt.title(title)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    save_plot(filename)
    plt.show()

plot_overlay(
    sine_data,
    sine_baseline_fake,
    "Overlay Comparison — Sine-Wave Baseline GAN",
    "08_overlay_sine_wave_baseline.png",
)

plot_overlay(
    sine_data,
    sine_improved_fake,
    "Overlay Comparison — Sine-Wave Improved GAN",
    "09_overlay_sine_wave_improved.png",
)

plot_overlay(
    spiral_data,
    spiral_baseline_fake,
    "Overlay Comparison — Spiral Baseline GAN",
    "10_overlay_spiral_baseline.png",
)

plot_overlay(
    spiral_data,
    spiral_improved_fake,
    "Overlay Comparison — Spiral Improved GAN",
    "11_overlay_spiral_improved.png",
)

In [ ]:
# ============================================================
# CELL 14 — MODEL COMPARISON TABLE FOR REPORT WRITING
# ============================================================
"""
Create a concise comparison table that is easy to quote in the report.
Lower mean/std distance suggests the generated distribution is closer
to the real data in terms of simple summary statistics.
"""

report_table = all_metrics[[
    "dataset_name",
    "model_name",
    "mean_l2_distance",
    "std_l2_distance",
    "training_time_seconds",
]].copy()

report_table["training_time_seconds"] = report_table["training_time_seconds"].round(2)
report_table["mean_l2_distance"] = report_table["mean_l2_distance"].round(4)
report_table["std_l2_distance"] = report_table["std_l2_distance"].round(4)

report_table.to_csv(CSV_DIR / "04_part1_report_ready_model_comparison.csv", index=False)

display(report_table.sort_values(["dataset_name", "model_name"]))

## Interpretation guide for the report

When you write the report, use the results from this notebook to discuss the following:

### 1. Sine-wave task
- Did the GAN recover the overall sinusoidal structure?
- Did the improved architecture produce tighter alignment with the real curve?
- Were the generated points noisy, scattered, or well concentrated?

### 2. Spiral task
- Was the spiral harder than the sine-wave?
- Did the baseline GAN struggle with curvature or partial coverage?
- Did the improved GAN better capture the wrapped structure?

### 3. Architecture modification
- Explain that the improved model increases generator capacity and replaces ReLU with LeakyReLU.
- Discuss whether this led to more stable or more realistic sample generation.

### 4. Limitations
- GAN losses do not always directly track visual quality.
- Simple summary-statistic distances are useful for 2D data but are not sufficient alone.
- Visual inspection remains important for synthetic low-dimensional tasks.

In [ ]:
# ============================================================
# CELL 15 — OPTIONAL QUICK CHECK OF SAVED FILES
# ============================================================
"""
List the saved outputs so that it is easy to verify what will be used
later in the report.
"""

print("Saved CSV files:")
for path in sorted(CSV_DIR.glob("*.csv")):
    print("-", path)

print("\nSaved plot files:")
for path in sorted(PLOTS_DIR.glob("*.png")):
    print("-", path)